<a href="https://colab.research.google.com/github/cactus1386/NationalCard-ImageProccessing/blob/main/trainOCR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as transforms
from PIL import Image
import os
import yaml

In [3]:
class CNNBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, padding=1):
        super(CNNBlock, self).__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size, padding=padding)
        self.bn = nn.BatchNorm2d(out_channels)

    def forward(self, x):
        x = self.conv(x)
        x = self.bn(x)
        x = nn.functional.relu(x)
        return x

In [4]:
class CRNN(nn.Module):
    def __init__(self, n_channels, num_classes, map2seq_in_dim, map2seq_out_dim, rnn_dim):
        super(CRNN, self).__init__()
        self.cnn = nn.ModuleDict({
            '0': CNNBlock(n_channels, 64),
            '2': CNNBlock(64, 128),
            '4': CNNBlock(128, 256),
            '5': CNNBlock(256, 256),
            '7': CNNBlock(256, 512),
            '8': CNNBlock(512, 512),
            '10': CNNBlock(512, 512, kernel_size=3, padding=1)
        })
        self.map2seq = nn.Linear(map2seq_in_dim, map2seq_out_dim)
        self.rnn1 = nn.LSTM(map2seq_out_dim, rnn_dim, bidirectional=True, batch_first=True)
        self.rnn2 = nn.LSTM(rnn_dim * 2, rnn_dim, bidirectional=True, batch_first=True)
        self.classifier = nn.Linear(rnn_dim * 2, num_classes)

    def forward(self, x):
        for key in sorted(self.cnn.keys(), key=lambda k: int(k)):
            x = self.cnn[key](x)
            if key in ['0', '2']:
                x = nn.functional.max_pool2d(x, 2)
            elif key in ['5', '8']:
                x = nn.functional.max_pool2d(x, (2, 1))
        batch, channels, height, width = x.size()
        x = x.permute(0, 3, 1, 2).reshape(batch, width, -1)
        x = self.map2seq(x)
        x, _ = self.rnn1(x)
        x, _ = self.rnn2(x)
        x = self.classifier(x)
        return x

In [5]:
def preprocess_image(image_path):
    with open("/content/drive/MyDrive/crnn-fa-printed-96-long/preprocessor/image_processor_config.yaml", "r") as f:
        config = yaml.safe_load(f)

    transform = transforms.Compose([
        transforms.Grayscale(num_output_channels=1),
        transforms.Resize((32, 384)),
        transforms.Lambda(lambda x: x.transpose(Image.FLIP_LEFT_RIGHT) if config["mirror"] else x),
        transforms.ToTensor(),
        transforms.Normalize(mean=config["mean"], std=config["std"]),
        transforms.Lambda(lambda x: x * config["rescale"])
    ])
    image = Image.open(image_path).convert("L")
    return transform(image)

In [3]:
class NewOCRDataset(Dataset):
    def __init__(self, image_dir):
        self.image_dir = image_dir
        self.image_paths = [os.path.join(image_dir, fname) for fname in os.listdir(image_dir) if fname.endswith(('.png', '.jpg', '.jpeg'))]

        with open("/content/drive/MyDrive/crnn-fa-printed-96-long/model_config.yaml", "r") as f:
            config = yaml.safe_load(f)
        self.char2id = {char: idx for idx, char in config["id2label"].items()}

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = preprocess_image(img_path)

        label = os.path.splitext(os.path.basename(img_path))[0]
        label = format_label(label)

        label_ids = [self.char2id.get(c, 0) for c in label]
        return image, torch.tensor(label_ids, dtype=torch.long), len(label_ids)

NameError: name 'Dataset' is not defined

In [7]:
def collate_fn(batch):
    images, targets, target_lengths = zip(*batch)
    images = torch.stack(images, dim=0)
    targets = [t.clone().detach() for t in targets]
    target_lengths = torch.tensor(target_lengths, dtype=torch.long)
    return images, targets, target_lengths

In [8]:
def train_model(model, dataloader, num_epochs=5):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    criterion = nn.CTCLoss(blank=0, zero_infinity=True)
    optimizer = optim.Adam(model.parameters(), lr=0.0001)

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0
        for batch_idx, (images, targets, target_lengths) in enumerate(dataloader):
            images = images.to(device)
            targets = torch.cat(targets).to(device)
            target_lengths = target_lengths.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            outputs = outputs.log_softmax(2)

            # CTC Loss انتظار دارد که خروجی‌ها به شکل (T, N, C) باشند
            # خروجی مدل ما (N, T, C) است، پس باید آن را permute کنیم
            outputs = outputs.permute(1, 0, 2)

            # طول ورودی‌ها (input_lengths) برابر با طول دنباله خروجی مدل است
            input_lengths = torch.full((images.size(0),), outputs.size(0), dtype=torch.long).to(device)
            loss = criterion(outputs, targets, input_lengths, target_lengths)

            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            print(f"Epoch [{epoch+1}/{num_epochs}], Batch [{batch_idx}], Loss: {loss.item():.4f}")

        avg_loss = total_loss / len(dataloader)
        print(f"Epoch [{epoch+1}/{num_epochs}] completed, Avg Loss: {avg_loss:.4f}")

    torch.save(model.state_dict(), "finetuned.pt")
    print("مدل ذخیره شد: finetuned.pt")

In [1]:
def format_label(label):
    if label.isdigit():
        if len(label) == 8:
            year = label[:4]
            month = label[4:6]
            day = label[6:]
            return f"{year}/{month}/{day}"
        elif len(label) == 10:
            return label
    return label

In [9]:
with open("/content/drive/MyDrive/crnn-fa-printed-96-long/model_config.yaml", "r") as f:
        config = yaml.safe_load(f)

model = CRNN(
    n_channels=config["n_channels"],
    map2seq_in_dim=config["map2seq_in_dim"],
    map2seq_out_dim=config["map2seq_out_dim"],
    rnn_dim=config["rnn_dim"],
    num_classes=len(config["id2label"])
)

model.load_state_dict(torch.load("/content/drive/MyDrive/crnn-fa-printed-96-long/model.pt"))
model.eval()

custom_data = {
    "/content/drive/MyDrive/images/ocrImage/code.png": "۹۴۹۰۵۴۹۰۰۸",
    "/content/drive/MyDrive/images/ocrImage/birth.png": "۱۳۶۷/۱۱/۲۳",
    "/content/drive/MyDrive/images/ocrImage/date.png": "۱۴۱۰/۰۳/۲۰",
    '/content/drive/MyDrive/images/ocrImage/father.png': 'اکبر',
    '/content/drive/MyDrive/images/ocrImage/last.png' : 'کرمی',
    "/content/drive/MyDrive/images/ocrImage/name.png": "تیرداد",
}

dataset = NewOCRDataset(custom_data)
dataloader = DataLoader(dataset, batch_size=2, shuffle=True, collate_fn=collate_fn)
train_model(model, dataloader, num_epochs=5)

<ipython-input-9-7af51136b1f8>:12: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("/content/drive/MyDrive/crnn-fa-printed-96-long/model.pt"))

Epoch [1/5], Batch [0], Loss: 7.3233
Epoch [1/5], Batch [1], Loss: 3.3941
Epoch [1/5], Batch [2], Loss: 7.5435
Epoch [1/5] completed, Avg Loss: 6.0870
Epoch [2/5], Batch [0], Loss: 2.0049
Epoch [2/5], Batch [1], Loss: 4.4200
Epoch [2/5], Batch [2], Loss: 3.3700
Epoch [2/5] completed, Avg Loss: 3.2650
Epoch [3/5], Batch [0], Loss: 2.4497
Epoch [3/5], Batch [1], Loss: 2.8471
Epoch [3/5], Batch [2], Loss: 2.0613
Epoch [3/5] completed, Avg Loss: 2.4527
Epoch [4/5], Batch [0], Loss: 0.5331
Epoch [4/5], Batch [1], Loss: 1.5538
Epoch [4/5], Batch [2], Loss: 1.1479
Epoch [4/5] completed, Avg Loss: 1.0783
Epoch [5/5], Batch [0], Loss: 0.9270
Epoch [5/5], Batch [1], Loss: 0.3821
Epoch [5/5], Batch [2], Loss: 4.7562
Epoch [5/5] completed, Avg Loss: 2.0218
مدل ذخیره شد: finetuned.pt
